# klygo.archive (operations.py): Modify and transform archives

This notebook favors runnable workflows over long reference tables. It uses only generated data inside a resettable temporary directory.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample data

In [1]:
from tempfile import TemporaryDirectory
from zipfile import BadZipFile
from klygo import archive, files

if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-archive-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-archive-c5au15li


In [2]:
source = files.mkdir(root / "dataset")
files.save(source / "labels.txt", ["cat", "dog"], verbose=False)
files.save(source / "config.json", {"batch": 16, "classes": 2}, verbose=False)
images = files.mkdir(source / "images")
files.save(images / "cat.txt", "image placeholder", verbose=False)

sample_archive = root / "dataset.zip"
archive.compress(source, sample_archive, verbose=False)
print("Members:", archive.list_files(sample_archive))

Members: ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


## Jump to an API

[`add`](#add) · [`remove`](#remove) · [`merge`](#merge) · [`split_by_size`](#split_by_size) · [`convert`](#convert) · [`recompress`](#recompress) · [`copy`](#copy)

<a id="add"></a>

## `archive.add`

Add files or directories to an existing archive.

**Signature:** `archive.add(archive_path: Union[str, pathlib.Path], files: Union[str, pathlib.Path, List[Union[str, pathlib.Path]]], verbose: bool = True, on_conflict: Literal['rename', 'overwrite', 'skip'] = 'rename') -> None`

Mutates a supported archive. `on_conflict` chooses rename, overwrite, or skip behavior for duplicate member names.

Each block below is runnable in order.

**Basic use**

In [3]:
extra = root / "extra.txt"
files.save(extra, "extra member", verbose=False)
archive.add(sample_archive, extra, verbose=False)
print(archive.list_files(sample_archive))

['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt', 'extra.txt']


**Practical variation**

In [4]:
archive.add(sample_archive, extra, on_conflict="rename", verbose=False)
print([name for name in archive.list_files(sample_archive) if "extra" in name])

['extra.txt', 'extra_dup1.txt']


**Expected error or edge behavior**

In [5]:
try:
    archive.add(sample_archive, extra, on_conflict="invalid", verbose=False)
except ValueError as error:
    print("ValueError:", error)

ValueError: on_conflict must be 'rename', 'overwrite', or 'skip'.


<a id="remove"></a>

## `archive.remove`

Remove named members from an existing archive.

**Signature:** `archive.remove(archive_path: Union[str, pathlib.Path], files: Union[str, List[str]]) -> None`

Rebuilds archives when the backend cannot delete members in place. Pass exact stored names from `list_files()`.

Each block below is runnable in order.

**Basic use**

In [6]:
remove_target = root / "remove.zip"
archive.copy(sample_archive, remove_target)
member = next(name for name in archive.list_files(remove_target) if name.endswith("labels.txt"))
archive.remove(remove_target, member)
print("Removed:", member not in archive.list_files(remove_target))

Removed: True


**Practical variation**

In [7]:
text_members = [name for name in archive.list_files(remove_target) if name.endswith(".txt")]
archive.remove(remove_target, text_members)
print(archive.list_files(remove_target))

['dataset/config.json']


**Expected error or edge behavior**

In [8]:
try:
    archive.remove(root / "missing.zip", "member.txt")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-c5au15li\\missing.zip'


<a id="merge"></a>

## `archive.merge`

Merge multiple archives into one destination archive.

**Signature:** `archive.merge(archive_paths: List[Union[str, pathlib.Path]], output_path: Union[str, pathlib.Path], overwrite: bool = False, verbose: bool = True) -> None`

Combines member trees into a new archive whose backend is chosen from the output extension.

Each block below is runnable in order.

**Basic use**

In [9]:
other_source = files.mkdir(root / "other")
files.save(other_source / "other.txt", "other", verbose=False)
other_archive = root / "other.zip"
archive.compress(other_source, other_archive, verbose=False)
merged = root / "merged.zip"
archive.merge([sample_archive, other_archive], merged, verbose=False)
print(archive.list_files(merged))

['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt', 'extra.txt', 'extra_dup1.txt', 'other/other.txt']


**Practical variation**

In [10]:
merged_tar = root / "merged.tar.gz"
archive.merge([sample_archive, other_archive], merged_tar, verbose=False)
print(archive.detect_format(merged_tar), len(archive.list_files(merged_tar)))

tar.gz 6


**Expected error or edge behavior**

In [11]:
try:
    archive.merge([sample_archive, other_archive], merged, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

FileExistsError: output_path already exists: C:\Users\Mine\AppData\Local\Temp\klygo-archive-c5au15li\merged.zip


<a id="split_by_size"></a>

## `archive.split_by_size`

Split an archive into size-limited archive parts.

**Signature:** `archive.split_by_size(archive_path: Union[str, pathlib.Path], size: Union[int, float], output_dir: Union[str, pathlib.Path] = '.', overwrite: bool = False, verbose: bool = True) -> List[str]`

Uses megabytes for `size` and returns generated part paths. Backend support differs by format.

Each block below is runnable in order.

**Basic use**

In [12]:
parts = archive.split_by_size(
    sample_archive,
    size=0.0001,
    output_dir=root / "parts",
    verbose=False,
)
print(parts)

['C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-c5au15li\\parts\\dataset_part_001.zip']


**Practical variation**

In [13]:
for part in parts:
    print(files.path(part).name, files.size(part), "bytes")

dataset_part_001.zip 659 bytes


**Expected error or edge behavior**

In [14]:
try:
    archive.split_by_size(sample_archive, size=0, output_dir=root / "invalid-parts", verbose=False)
except ValueError as error:
    print("ValueError:", error)

<a id="convert"></a>

## `archive.convert`

Convert an archive to another format.

**Signature:** `archive.convert(source_path: Union[str, pathlib.Path], target_path: Union[str, pathlib.Path], overwrite: bool = False, verbose: bool = True) -> None`

Extracts into a temporary directory and compresses with the target backend; it is not an in-place conversion.

Each block below is runnable in order.

**Basic use**

In [15]:
converted = root / "converted.tar.gz"
archive.convert(sample_archive, converted, verbose=False)
print(archive.detect_format(converted), archive.list_files(converted))

tar.gz ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt', 'extra.txt', 'extra_dup1.txt']


**Practical variation**

In [16]:
round_trip = root / "round-trip.zip"
archive.convert(converted, round_trip, verbose=False)
print(archive.compare(sample_archive, round_trip))

{'added_files': [], 'removed_files': [], 'common_files': ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt', 'extra.txt', 'extra_dup1.txt']}


**Expected error or edge behavior**

In [17]:
try:
    archive.convert(sample_archive, converted, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

FileExistsError: target_path already exists: C:\Users\Mine\AppData\Local\Temp\klygo-archive-c5au15li\converted.tar.gz. Use overwrite=True.


<a id="recompress"></a>

## `archive.recompress`

Rebuild an archive with a different compression level or format.

**Signature:** `archive.recompress(source_path: Union[str, pathlib.Path], target_path: Union[str, pathlib.Path], compresslevel: int = 6, overwrite: bool = False, verbose: bool = True) -> None`

Uses the same rebuild pipeline as conversion while exposing the target compression level.

Each block below is runnable in order.

**Basic use**

In [18]:
compact = root / "compact.zip"
archive.recompress(sample_archive, compact, compresslevel=9, verbose=False)
print(files.size(sample_archive), files.size(compact))

659 659


**Practical variation**

In [19]:
recompressed_tar = root / "recompressed.tar.gz"
archive.recompress(sample_archive, recompressed_tar, compresslevel=6, verbose=False)
print(archive.detect_format(recompressed_tar))

tar.gz


**Expected error or edge behavior**

In [20]:
try:
    archive.recompress(sample_archive, compact, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

FileExistsError: target_path already exists: C:\Users\Mine\AppData\Local\Temp\klygo-archive-c5au15li\compact.zip. Use overwrite=True.


<a id="copy"></a>

## `archive.copy`

Copy an archive without extracting or changing it.

**Signature:** `archive.copy(source_path: Union[str, pathlib.Path], target_path: Union[str, pathlib.Path], overwrite: bool = False) -> None`

Delegates filesystem copying and overwrite protection to `klygo.files.copy`; archive contents are not decoded.

Each block below is runnable in order.

**Basic use**

In [21]:
backup = root / "backup" / "dataset.zip"
archive.copy(sample_archive, backup)
print(backup, archive.compare(sample_archive, backup))

C:\Users\Mine\AppData\Local\Temp\klygo-archive-c5au15li\backup\dataset.zip {'added_files': [], 'removed_files': [], 'common_files': ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt', 'extra.txt', 'extra_dup1.txt']}


**Practical variation**

In [22]:
archive.copy(sample_archive, backup, overwrite=True)
print("Replaced intentionally:", archive.test(backup))

Replaced intentionally: True


**Expected error or edge behavior**

In [23]:
try:
    archive.copy(sample_archive, backup)
except FileExistsError as error:
    print("FileExistsError:", error)

FileExistsError: Target already exists: C:\Users\Mine\AppData\Local\Temp\klygo-archive-c5au15li\backup\dataset.zip


## Put it together

In [24]:
working_zip = root / "workflow.zip"
archive.copy(sample_archive, working_zip, overwrite=True)
extra_note = root / "README.txt"
files.save(extra_note, "release notes", overwrite=True, verbose=False)
archive.add(working_zip, extra_note, on_conflict="overwrite", verbose=False)
converted_tar = root / "workflow.tar.gz"
archive.convert(working_zip, converted_tar, overwrite=True, verbose=False)
print("Converted members:", archive.list_files(converted_tar))

Converted members: ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt', 'extra.txt', 'extra_dup1.txt', 'README.txt']


## Remember

- Archive mutations (`add`, `remove`, `merge`, `convert`) rebuild archives safely via temporary files and support `on_conflict` policies (`skip`, `overwrite`, `rename`).
- `split_by_size()` and `add(..., on_conflict='rename')` preserve compound extensions such as `.tar.gz` automatically via `klygo.files`.
- Next in `klygo.archive`: continue with [Document_Archive_Context.ipynb](Document_Archive_Context.ipynb) ([Colab](https://colab.research.google.com/drive/1ulSJac2OViJlPAeZVYw8YvQZXA1a3tP2?usp=sharing)) or return to [Document_Archive_Overview.ipynb](Document_Archive_Overview.ipynb) ([Colab](https://colab.research.google.com/drive/1z5cuH35jFOkTtSkS4IcufF8ZygnylEJS?usp=sharing)).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).